# Notebook 14: Performance Optimization
**Filename:** `14_Performance_Optimization.ipynb`  
**Topics Covered:** Vectorization, `apply()` vs Loops, Memory Optimization, Category Data Type, Efficient Pandas Practices

---

## 1. Vectorization vs Loops & `apply()`

### Concept Explanation
Vectorization executes mathematical operations across entire arrays simultaneously using optimized C-level code underneath. Standard Python `for` loops and Pandas `.apply()` execute row-by-row in Python overhead space, making them significantly slower.

### Real-world Example
Calculating distance values across thousands of GPS coordinate pairs.

### Business Example
Applying sales tax rates across 1,000,000 transaction records.

### AI/ML Example
Computing feature transformations during large-scale dataset prep.

In [1]:
import pandas as pd
import numpy as np
import time

# Benchmark dataset
n = 500000
df_bench = pd.DataFrame({'A': np.random.randn(n), 'B': np.random.randn(n)})

# 1. Slow Python Loop
start = time.time()
loop_res = []
for index, row in df_bench.iterrows():
    loop_res.append(row['A'] + row['B'])
loop_time = time.time() - start

# 2. Vectorized NumPy/Pandas
start = time.time()
vec_res = df_bench['A'] + df_bench['B']
vec_time = time.time() - start

print(f"Iterrows Loop Time: {loop_time:.4f} seconds")
print(f"Vectorized Time:    {vec_time:.4f} seconds")
print(f"Speedup Factor:     {loop_time / vec_time:.1f}x faster")

Iterrows Loop Time: 57.9622 seconds
Vectorized Time:    0.0120 seconds
Speedup Factor:     4836.8x faster


---

## 2. Memory Optimization (`category` & Downcasting)

### Concept Explanation
Object/string columns with low cardinality consume excessive memory. Converting these columns to the `category` data type stores values as integer pointers, drastically reducing memory usage. Downcasting float/int types further shrinks footprint.

### Real-world Example
Optimizing high-frequency sensor status columns (`'ON'`, `'OFF'`, `'IDLE'`).

### Business Example
Shrinking state code columns (`'NY'`, `'CA'`) across multi-gigabyte customer logs.

### AI/ML Example
Reducing dataset size to fit massive feature tables into system RAM.

In [2]:
import pandas as pd
import numpy as np

# Sample dataset with repeated string categories
df_mem = pd.DataFrame({
    'Status': np.random.choice(['Pending', 'Approved', 'Rejected'], size=100000),
    'Val': np.random.randint(1, 100, size=100000)
})

mem_before = df_mem.memory_usage(deep=True).sum() / (1024**2)

# Convert string object to category type
df_mem['Status'] = df_mem['Status'].astype('category')

mem_after = df_mem.memory_usage(deep=True).sum() / (1024**2)

print(f"Memory Before: {mem_before:.2f} MB")
print(f"Memory After:  {mem_after:.2f} MB")
print(f"Reduction:     {((mem_before - mem_after) / mem_before) * 100:.1f}%")

Memory Before: 1.88 MB
Memory After:  0.48 MB
Reduction:     74.6%


---

## 3. Efficient Pandas Practices

### Concept Explanation
Key best practices for maximizing performance:
1. **Avoid Iterrows:** Rely on vectorized expression logic or NumPy arrays.
2. **Use Categoricals:** Cast repeating text attributes to `category`.
3. **Filter Early:** Apply filtering masks early to trim rows before complex joins/aggregations.
4. **Use In-place Alternatives / Selection:** Specify exact `usecols` when reading CSV files via `pd.read_csv()`.

### Real-world Example
Loading specific columns from multi-GB log exports rather than pulling full files.

### Business Example
Structuring daily analytical ETL pipelines for maximum execution efficiency.

### AI/ML Example
Optimizing data pipelines to maximize throughput during model training loops.

In [3]:
import pandas as pd

# Demonstrating query filtering before aggregation
df_perf = pd.DataFrame({
    'Region': np.random.choice(['East', 'West'], size=100000),
    'Sales': np.random.randint(10, 500, size=100000)
})

# Efficient pattern: Filter rows FIRST, then aggregate target column
result = df_perf[df_perf['Region'] == 'East']['Sales'].sum()
print("Filtered Sum Result:", result)

Filtered Sum Result: 12702172


---

## Minimum 5 Interview Questions with Answers

1. **Why is vectorization so much faster than Python `for` loops in Pandas?**  
   * **Answer:** Vectorization runs compiled C-level loops over contiguous memory blocks, avoiding Python interpreter overhead, dynamic type checking, and object boxing/unboxing during execution.

2. **When should you convert a string object column to the `category` data type?**  
   * **Answer:** Convert when a string column contains low cardinality (a small count of unique repeating strings relative to row volume, e.g., gender, order status, region codes).

3. **Why is `df.iterrows()` considered an anti-pattern for performance?**  
   * **Answer:** `df.iterrows()` creates a Pandas Series object for every single row iteratively, introducing immense Python object creation overhead and running dramatically slower than vectorized ops.

4. **How can you optimize memory when reading huge CSV files with `pd.read_csv()`?**  
   * **Answer:** Specify `usecols` to load only target columns, set efficient data types via `dtype`, or stream data in smaller iteration batches using the `chunksize` parameter.

5. **Does `df.apply()` perform true vectorization?**  
   * **Answer:** No. While cleaner syntactically, `apply()` internally iterates over rows/columns in Python space and does not leverage C-speed vectorization.

---

## Self Reflection
* **What I Learned:** Mastered vectorized computation benchmarks, memory optimization using `category` types, integer/float downcasting, and high-performance pipeline practices.
* **Key Takeaway:** Writing vectorized operations and optimizing data types reduces runtime execution from minutes to milliseconds while significantly lowering memory consumption.